# 01 — Data assembly

**Mini Project 2.** Loads the four inputs, joins them to one row per SA2,
attaches geometry and computes density, then writes a single tidy analysis
table for the later notebooks.

Runs top to bottom on a fresh clone. Nothing here is analysis — it is the
plumbing, and it should be dull.

## The question

Written down before touching the data:

> Can the socio-economic and housing profile of a small area tell us how
> crowded its households are — and do the natural groupings of small areas
> outperform the standard administrative ones at that job?

Everything below either supports that question or gets out of its way.

## Setup

In [ ]:
import sys
sys.path.append("../src")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import mp2

pd.set_option("display.max_columns", 60)
pd.set_option("display.width", 160)

print(f"data     -> {mp2.DATA}")
print(f"outputs  -> {mp2.OUT}")
print(f"NZTM EPSG-> {mp2.NZTM}")

## Load the raw inputs

Print the columns of each one **before** writing any join. The Explorer names
columns per table and they will not match what you expect, so map them
explicitly rather than guessing.

In [ ]:
households = mp2.load_raw(mp2.HOUSEHOLDS)
individuals = mp2.load_raw(mp2.INDIVIDUALS)
concordance = mp2.load_raw(mp2.CONCORDANCE)

print("HOUSEHOLDS")
print(list(households.columns))
print()
print("INDIVIDUALS")
print(list(individuals.columns))
print()
print("CONCORDANCE")
print(list(concordance.columns))

In [ ]:
mp2.snapshot(households, "households (raw)")
mp2.snapshot(individuals, "individuals (raw)")
mp2.snapshot(concordance, "concordance (raw)")

## TODO — map the columns

Before the join works you need to tell `mp2.assemble` which columns are the
year-and-area label and which are the measures. Write the mapping here as an
explicit dict, so a reader can see exactly which published variable became
which column in your frame.

- [ ] Rename measure columns to short snake_case names of your choosing
- [ ] Record the original published label for each one (README table)
- [ ] Note which are level 1 vs level 2 of their classification

In [ ]:
# TODO: replace with your real mapping, then adapt mp2.assemble() to match.
RENAME = {
    # "Census 2023 households in occupied private dwellings": "households",
    # "Household crowding index level 1": "crowding",
    # ... etc
}

# households = households.rename(columns=RENAME)

## Join the three inputs

Households + individuals + concordance, on the SA2 code. One row per SA2.

In [ ]:
# frame = mp2.assemble(households, individuals, concordance, year=mp2.CENSUS_YEAR)
# mp2.snapshot(frame, "assembled frame")

## Attach geometry and compute density

Project to NZTM (EPSG:2193) **before** any area calculation — degrees are not
metres, and area in EPSG:4326 is meaningless. `mp2.attach_geometry` does the
projection, drops the null-geometry SA2s and prints what it dropped.

In [ ]:
# gdf = mp2.attach_geometry(frame)
# print(gdf.crs)
# print(gdf[["sa2_code", "area_km2", "density_per_km2"]].describe())

## Sanity checks

Every one of these should print a number you can defend.

- [ ] Row count matches the published SA2 count, minus the null-geometry ones
- [ ] No duplicate `sa2_code`
- [ ] Codes are 7 digits and join cleanly to the boundary file
- [ ] Area values plausible (NZ is ~268,000 km²)
- [ ] Nothing in the frame is a string where a number belongs

In [ ]:
# TODO: run the checks above and print each result

# print("duplicate sa2_code:", frame["sa2_code"].duplicated().sum())
# print("code length:", frame["sa2_code"].str.len().value_counts().to_dict())
# print("total area km2:", round(gdf["area_km2"].sum(), 0))

## Suppression summary

Fixed random rounding to base 3 plus suppression of sensitive cells below six
means small areas arrive with holes. **Suppression is not missing-at-random**
— it concentrates in the smallest areas. Quantify it here, because it feeds
straight into the limitations section of the README.

In [ ]:
# TODO: replace the column list with your own measure columns
# report = mp2.suppression_report(frame)
# print(report.to_string(index=False))

## TODO — is suppression biased?

This is the part most students skip and yours should not. If suppressed cells
concentrate in small or rural areas, the clustering is biased toward large
urban SA2s and every downstream result inherits that.

- [ ] Correlate per-SA2 suppression share against area_km2 and density
- [ ] Plot it, or print a crosstab of suppression against urban/rural
- [ ] Write the answer down in a markdown cell — one sentence, with the number

In [ ]:
# share = mp2.suppressed_share(gdf)
# print(share.describe())
# print("corr with area:", round(share.corr(gdf["area_km2"]), 3))

# TODO: your answer goes in the markdown cell above, not in a comment

## Save the analysis table

`outputs/analysis_frame.csv` is what notebooks 02–05 read. Save the
non-spatial version as CSV so the later notebooks do not need geopandas;
keep the GeoDataFrame as a GeoPackage for mapping.

In [ ]:
# gdf.drop(columns="geometry").to_csv(mp2.OUT / "analysis_frame.csv", index=False)
# gdf.to_file(mp2.OUT / "analysis_frame.gpkg", driver="GPKG")
# print("written:", len(gdf), "rows")

## Notes for the README

*[Record here, as you go: the exact download path and access date, the
licence line, how many SA2s you dropped and why, and any column you chose
not to use. This is copy-paste material for the README rather than something
you reconstruct later from memory.]*